# 02. 모델링 & 실험 관리
## Credit Card Fraud Detection
---
**실험 설계**: 4개 모델 × SMOTE 적용 여부 비교  
**실험 추적**: MLflow (재현 가능한 실험 관리)  
**평가 지표**: Average Precision (불균형 데이터 최적)  
**임계값 전략**: F1 최대화 기반 동적 임계값


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.utils import load_config, set_seed, make_dirs
from src.data_loader import load_data
from src.preprocess import preprocess, split_data
from src.train import train_all
from src.evaluate import evaluate, compare_models
from src.visualize import plot_roc_pr_curves, plot_model_comparison, plot_confusion_matrix, plot_threshold_analysis

cfg = load_config('../configs/config.yaml')
set_seed(cfg['project']['seed'])
make_dirs(cfg)

print("실험 설정 로드 완료")
print(f"MLflow 실험: {cfg['mlflow']['experiment_name']}")
print(f"불균형 처리: {cfg['imbalance']['strategy'].upper()}")
print(f"주 평가 지표: {cfg['evaluation']['primary_metric']}")

In [ ]:
# ── 데이터 로드 & 전처리 ────────────────────────────────────
df_raw = load_data(cfg)

# 중복 제거
n_before = len(df_raw)
df_raw = df_raw.drop_duplicates()
print(f"중복 제거: {n_before - len(df_raw)}건")

# 피처 엔지니어링 (Amount_log, Hour, IsNight)
df = preprocess(df_raw, cfg)
print(f"\n최종 피처 수: {df.shape[1] - 1}개")
print(f"피처 목록: {[c for c in df.columns if c != cfg['data']['target_col']]}")

In [ ]:
# ── Train / Val / Test 분리 ─────────────────────────────────
X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = split_data(df, cfg)

print(f"\n피처 수: {len(feature_cols)}")
print(f"사기 비율 - Train: {y_train.mean():.4%}  Val: {y_val.mean():.4%}  Test: {y_test.mean():.4%}")
print("\nStratified split으로 각 세트의 사기 비율이 동일하게 유지됨")

In [ ]:
# ── 4개 모델 학습 (MLflow 실험 추적) ───────────────────────
print("MLflow 실험 시작...")
print("각 실험의 파라미터/지표/모델이 mlruns/ 폴더에 자동 저장됩니다\n")

results = train_all(X_train, y_train, X_val, y_val, cfg)
print("\n전체 학습 완료")

In [ ]:
# ── 검증 세트 성능 비교 ─────────────────────────────────────
compare_df = compare_models(results)
print("\n모델 성능 비교 (검증 세트 기준):")
print(compare_df[['Model', 'ROC-AUC', 'Avg Precision', 'F1', 'Precision', 'Recall']].to_string(index=False))
print(f"\n★ Average Precision 기준 최고 모델: {compare_df.iloc[0]['Model']}")

In [ ]:
# ── 최고 모델로 테스트 세트 최종 평가 ─────────────────────
best_name = compare_df.iloc[0]['Model']
best_pipe  = results[best_name]['model']
best_threshold = results[best_name]['threshold']

print(f"최종 모델: {best_name}")
print(f"검증 세트 최적 임계값: {best_threshold:.4f}")
print("\n[테스트 세트 최종 평가]")

y_prob_test = best_pipe.predict_proba(X_test)[:, 1]
test_metrics = evaluate(y_test, y_prob_test, threshold=best_threshold)

# 결과 저장 (노트북 간 공유)
import json
os.makedirs('../outputs/reports', exist_ok=True)
with open('../outputs/reports/test_metrics.json', 'w') as f:
    json.dump({'model': best_name, **test_metrics}, f, indent=2)
print("\n테스트 결과 저장: outputs/reports/test_metrics.json")

In [ ]:
# ── ROC & PR 커브 ───────────────────────────────────────────
# 테스트 세트 확률값 저장
for name, res in results.items():
    res['X_test'] = X_test

fig = plot_roc_pr_curves(results, y_test,
                          save_path='../outputs/figures/06_roc_pr.png')
plt.show()
print("\n불균형 데이터에서 ROC-AUC는 과대평가됨")
print("   PR Curve의 Average Precision이 더 신뢰할 수 있는 지표입니다")

In [ ]:
# ── 모델 비교 바 차트 ────────────────────────────────────────
fig = plot_model_comparison(compare_df,
                             save_path='../outputs/figures/07_model_comparison.png')
plt.show()

In [ ]:
# ── Confusion Matrix ────────────────────────────────────────
y_pred_test = (y_prob_test >= best_threshold).astype(int)
fig = plot_confusion_matrix(y_test, y_pred_test, model_name=best_name,
                             save_path='../outputs/figures/08_confusion_matrix.png')
plt.show()

tn = ((y_pred_test==0)&(y_test==0)).sum()
fp = ((y_pred_test==1)&(y_test==0)).sum()
fn = ((y_pred_test==0)&(y_test==1)).sum()
tp = ((y_pred_test==1)&(y_test==1)).sum()

print(f"\n비즈니스 임팩트 해석:")
print(f"사기 탐지 성공 (TP): {tp}건 → 피해 예방")
print(f"사기 미탐 (FN):      {fn}건 → 피해 발생")
print(f"정상 오탐 (FP):     {fp}건 → 고객 불편")
print(f"정상 정상처리 (TN):  {tn:,}건")

In [ ]:
# ── 임계값 분석 ─────────────────────────────────────────────
fig = plot_threshold_analysis(y_test, y_prob_test,
                               save_path='../outputs/figures/09_threshold.png')
plt.show()
print("""
비즈니스 상황에 따른 임계값 선택:
  - 고객 불편 최소화 → 높은 Precision 우선 (임계값 높게)
  - 사기 피해 최소화 → 높은 Recall 우선  (임계값 낮게)
  - 균형 → F1 최대화 임계값 (현재 선택)
""")

In [ ]:
# ── MLflow UI 안내 ──────────────────────────────────────────
print("="*55)
print("  MLflow 실험 결과 확인 방법")
print("="*55)
print("""
터미널에서 실행:
  $ cd credit-fraud-mlpipeline
  $ mlflow ui

브라우저에서:
  http://localhost:5000

확인 가능한 정보:
  - 각 모델의 파라미터 / 지표 비교
  - 실험 재현 (run ID로 동일 결과 재현)
  - 모델 아티팩트 다운로드
""")